# Run the Streamlit Viewer in Colab

Use this notebook after Parts 2 and 3 have produced the B08 GeoTIFF and `autorift_scatter_demeaned_*.npz` files. It starts `4. autorift_speed_scatter_app.py` inside Colab and opens it through a temporary Cloudflare URL.

Keep this notebook in the same folder as `4. autorift_speed_scatter_app.py` and the Part 3 outputs.


## 1. Install viewer dependencies

This is only for the interactive viewer. It does not install autoRIFT because Part 3 has already produced the `.npz` result.


In [1]:
from __future__ import annotations

import importlib.util
import subprocess
import sys

PACKAGE_FOR_MODULE = {
    "streamlit": "streamlit",
    "plotly": "plotly",
    "rasterio": "rasterio",
    "PIL": "Pillow",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
}
missing = [pkg for mod, pkg in PACKAGE_FOR_MODULE.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
print("Viewer dependencies are ready.")


Viewer dependencies are ready.


## 2. Check files

The Streamlit app needs the `.py` file, the Part 3 `.npz`, and preferably the Part 2 B08 GeoTIFF for the basemap.


In [2]:
from pathlib import Path

LAB_DIR = Path.cwd().resolve()
APP = LAB_DIR / "4. autorift_speed_scatter_app.py"
npz_files = sorted(LAB_DIR.glob("autorift_scatter_demeaned_*.npz"))
basemap = LAB_DIR / "B08_fox_2026-03-01.tif"

if not APP.is_file():
    raise FileNotFoundError(APP)
if not npz_files:
    raise FileNotFoundError("No autorift_scatter_demeaned_*.npz found. Run Part 3 first.")

print(f"Working folder: {LAB_DIR}")
print(f"App: {APP.name}")
print("NPZ files:")
for f in npz_files:
    print(" -", f.name)
print(f"Basemap: {basemap.name if basemap.is_file() else '(not found; app still runs without basemap)'}")


Working folder: /home/xguo/Remote-Sense-Application/FertureTrackingLab
App: 4. autorift_speed_scatter_app.py
NPZ files:
 - autorift_scatter_demeaned_2026-03-01__2026-03-21.npz
Basemap: B08_fox_2026-03-01.tif


## 3. Start Streamlit and create a public URL

Run this cell and wait for the `https://...trycloudflare.com` link. Open that link in a new browser tab. The link is temporary and works only while this Colab runtime is active.


In [3]:
import os
import re
import stat
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

PORT = 8501
IN_COLAB = "google.colab" in sys.modules

# Stop old processes from a previous run in this same runtime/session.
subprocess.run(["pkill", "-f", "streamlit run"], check=False)
subprocess.run(["pkill", "-f", "cloudflared tunnel"], check=False)
time.sleep(1)

streamlit_log = open("streamlit_colab.log", "w")
streamlit_proc = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "streamlit",
        "run",
        str(APP),
        "--server.port",
        str(PORT),
        "--server.address",
        "0.0.0.0",
        "--server.headless",
        "true",
        "--browser.gatherUsageStats",
        "false",
    ],
    stdout=streamlit_log,
    stderr=subprocess.STDOUT,
)

time.sleep(4)
if streamlit_proc.poll() is not None:
    streamlit_log.close()
    print(Path("streamlit_colab.log").read_text(errors="replace")[-3000:])
    raise RuntimeError("Streamlit stopped early. See log above.")

if not IN_COLAB:
    print("Streamlit is running locally.")
    print(f"Open this URL in your browser: http://localhost:{PORT}")
else:
    print("Streamlit is running in Colab. Creating Cloudflare tunnel...")
    cloudflared = Path("/content/cloudflared")
    cloudflared.parent.mkdir(parents=True, exist_ok=True)

    if not cloudflared.is_file():
        url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
        urllib.request.urlretrieve(url, cloudflared)
        cloudflared.chmod(cloudflared.stat().st_mode | stat.S_IEXEC)

    tunnel_proc = subprocess.Popen(
        [str(cloudflared), "tunnel", "--url", f"http://localhost:{PORT}"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    public_url = None
    for _ in range(120):
        line = tunnel_proc.stdout.readline()
        if line:
            print(line.rstrip())
            m = re.search(r"https://[-a-zA-Z0-9.]+trycloudflare.com", line)
            if m:
                public_url = m.group(0)
                break
        time.sleep(0.5)

    if not public_url:
        raise RuntimeError("Could not find a Cloudflare URL. Rerun this cell, or check the output above.")

    print("\nOpen this Streamlit app URL:")
    print(public_url)


Streamlit is running locally.
Open this URL in your browser: http://localhost:8501


## 4. Stop the app

Run this only when you are done with the viewer.


In [8]:
import subprocess

subprocess.run(["pkill", "-f", "streamlit run"], check=False)
subprocess.run(["pkill", "-f", "cloudflared tunnel"], check=False)
print("Stopped Streamlit and Cloudflare tunnel processes.")


Stopped Streamlit and Cloudflare tunnel processes.
